In [11]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
import json
import time
import re

options = Options()
driver = webdriver.Chrome(options=options)

# Vào trang winmart
driver.get("https://www.winmart.vn")
time.sleep(5)  # Chờ load hết menu

# Lấy tất cả link có dạng /products/ hoặc category
links = driver.find_elements(By.TAG_NAME, "a")

slugs = set()
for link in links:
    href = link.get_attribute("href") or ""
    # Lấy slug danh mục dạng --cXX
    if re.search(r'--c\d+', href):
        slug = href.split("/")[-1]
        slugs.add(slug)

print(f"Tìm được {len(slugs)} danh mục:")
for s in sorted(slugs):
    print(s)

# Lưu file
with open('all_slugs.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(sorted(slugs)))

print("✅ Đã lưu vào all_slugs.txt")
driver.quit()


Tìm được 93 danh mục:
banh-bao--c01156?storeCode=1535
banh-keo--c07?storeCode=1535
banh-mi--c0115?storeCode=1535
banh-snack--c0129?storeCode=1535
banh-xop-banh-quy--c0127?storeCode=1535
bia--c01135?storeCode=1535
binh-xit-con-trung--c01139?storeCode=1535
bo-sua-pho-mai--c0138?storeCode=1535
bot-cac-loai--c0125?storeCode=1535
ca-bo-vien--c01160?storeCode=1535
ca-phe--c01162?storeCode=1535
cha-gio--c01159?storeCode=1535
cham-soc-be--c12?storeCode=1535
cham-soc-ca-nhan--c11?storeCode=1535
cham-soc-ca-nhan-cho-be--c0155?storeCode=1535
cham-soc-ca-nhan-khac--c0149?storeCode=1535
cham-soc-da--c0146?storeCode=1535
cham-soc-phu-nu--c0148?storeCode=1535
cham-soc-rang-mieng--c0147?storeCode=1535
cham-soc-toc--c0145?storeCode=1535
chao--c01146?storeCode=1535
cu-qua--c01168?storeCode=1535
dau-an--c01149?storeCode=1535
dau-hu--c01138?storeCode=1535
dien-gia-dung--c26?storeCode=1535
do-dung-gia-dinh--c25?storeCode=1535
do-dung-nha-bep--c01133?storeCode=1535
do-dung-phong-ngu--c0197?storeCode=1535
do

In [1]:
import requests
import json
import time
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

def create_session():
    session = requests.Session()
    
    # Tự động retry 3 lần nếu lỗi
    retry = Retry(
        total=3,
        backoff_factor=1,       # Chờ 1s, 2s, 4s giữa các lần retry
        status_forcelist=[429, 500, 502, 503, 504]  # Retry khi gặp các lỗi này
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)
    
    session.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "Accept": "application/json",
        "Connection": "keep-alive"  # Giữ kết nối tái sử dụng
    })
    return session

session = create_session()

def crawl_winmart(slug):
    all_items = []
    page = 1

    while True:
        url = f"https://api-crownx.winmart.vn/it/api/web/v3/item/category?orderByDesc=true&pageNumber=1&pageSize=8&slug={slug}&storeGroupCode=1998"

        res = session.get(url, timeout=(5, 30))  # 5s connect, 30s read
        data = res.json()

        items = data["data"]["items"]
        all_items.extend(items)
        total_pages = data["paging"]["totalPages"]

        print(f"[{slug}] Trang {page}/{total_pages} - Tổng: {len(all_items)} sản phẩm")

        if page >= total_pages:
            break

        page += 1
        time.sleep(0.3)

    with open(f"{slug}.json", "w", encoding="utf-8") as f:
        json.dump(all_items, f, ensure_ascii=False, indent=2)

    print(f"✅ Xong! Đã lưu {len(all_items)} sản phẩm vào {slug}.json")
    return all_items

# Đọc từng slug trong file và chạy
with open("slug.txt", "r", encoding="utf-8") as f:
    slugs = [line.strip() for line in f if line.strip()]

print(f"📋 Tìm thấy {len(slugs)} danh mục")

failed = []

for i, slug in enumerate(slugs, 1):
    print(f"\n[{i}/{len(slugs)}] Đang cào: {slug}")
    try:
        crawl_winmart(slug)
    except Exception as e:
        print(f"⚠️ Bỏ qua [{slug}]: {e}")
        failed.append(slug)
    time.sleep(0.5)

print(f"\n{'='*40}")
print(f"✅ Thành công: {len(slugs) - len(failed)}/{len(slugs)} danh mục")
if failed:
    print(f"❌ Thất bại ({len(failed)} danh mục):")
    for s in failed:
        print(f"   - {s}")
    with open("slug_failed.txt", "w", encoding="utf-8") as f:
        f.write('\n'.join(failed))
    print(f"\n💾 Đã lưu slug lỗi vào slug_failed.txt")

📋 Tìm thấy 92 danh mục

[1/92] Đang cào: banh-bao--c01156&storeCode=1535
[banh-bao--c01156&storeCode=1535] Trang 1/2 - Tổng: 8 sản phẩm
[banh-bao--c01156&storeCode=1535] Trang 2/2 - Tổng: 16 sản phẩm
✅ Xong! Đã lưu 16 sản phẩm vào banh-bao--c01156&storeCode=1535.json

[2/92] Đang cào: banh-keo--c07&storeCode=1535
[banh-keo--c07&storeCode=1535] Trang 1/59 - Tổng: 8 sản phẩm
[banh-keo--c07&storeCode=1535] Trang 2/59 - Tổng: 16 sản phẩm
[banh-keo--c07&storeCode=1535] Trang 3/59 - Tổng: 24 sản phẩm
[banh-keo--c07&storeCode=1535] Trang 4/59 - Tổng: 32 sản phẩm
[banh-keo--c07&storeCode=1535] Trang 5/59 - Tổng: 40 sản phẩm
[banh-keo--c07&storeCode=1535] Trang 6/59 - Tổng: 48 sản phẩm
[banh-keo--c07&storeCode=1535] Trang 7/59 - Tổng: 56 sản phẩm
[banh-keo--c07&storeCode=1535] Trang 8/59 - Tổng: 64 sản phẩm
[banh-keo--c07&storeCode=1535] Trang 9/59 - Tổng: 72 sản phẩm
[banh-keo--c07&storeCode=1535] Trang 10/59 - Tổng: 80 sản phẩm
[banh-keo--c07&storeCode=1535] Trang 11/59 - Tổng: 88 sản phẩm
[b

KeyboardInterrupt: 

In [1]:
import json
import pandas as pd
import os
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

# 📁 Thư mục chứa các file JSON
folder_path = "D:\\a"  # ← đổi thành folder của bạn

all_rows = []

# 🔥 Đọc toàn bộ file JSON
for filename in os.listdir(folder_path):
    if filename.endswith(".json"):
        filepath = os.path.join(folder_path, filename)

        # ✅ Lấy tên danh mục từ tên file (bỏ đuôi .json)
        category_name = filename.replace(".json", "")

        with open(filepath, encoding='utf-8') as f:
            data = json.load(f)

        if not data:  # bỏ qua file rỗng
            continue

        df = pd.DataFrame(data)

        for _, row in df.iterrows():
            seo = row.get('seoName', '')
            url = f"https://www.winmart.vn/products/{seo}" if seo else ''

            all_rows.append([
                category_name,                      # ✅ Cột danh mục
                row.get('name', ''),
                row.get('shortDescription', ''),
                row.get('uomName', ''),
                row.get('price', ''),
                row.get('salePrice', ''),
                row.get('quantity', ''),
                row.get('scaleQuantity', ''),
                url
            ])

print(f"📦 Tổng sản phẩm đọc được: {len(all_rows)}")

# 📊 Tạo Excel
wb = Workbook()
ws = wb.active
ws.title = "All Products"

# Header
headers = ['Danh mục', 'Tên sản phẩm', 'Mô tả ngắn', 'Đơn vị', 'Giá gốc', 'Giá sale', 'Tồn kho', 'Scale Qty', 'Link']
ws.append(headers)

# Style header - nền xanh, chữ trắng, in đậm
header_fill = PatternFill(start_color="1F4E79", end_color="1F4E79", fill_type="solid")
header_font = Font(bold=True, color="FFFFFF", size=11)

for cell in ws[1]:
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = Alignment(horizontal="center", vertical="center")

ws.row_dimensions[1].height = 20

# Ghi dữ liệu
for row_data in all_rows:
    ws.append(row_data)

    last_row = ws.max_row
    url = row_data[-1]  # cột Link là cột cuối

    # Tạo hyperlink ở cột "Tên sản phẩm" (cột B = 2)
    name_cell = ws.cell(row=last_row, column=2)
    if url:
        name_cell.hyperlink = url
        name_cell.font = Font(color="0563C1", underline="single")

# Chỉnh độ rộng cột
column_widths = {
    'A': 25,   # Danh mục
    'B': 40,   # Tên sản phẩm
    'C': 35,   # Mô tả ngắn
    'D': 12,   # Đơn vị
    'E': 12,   # Giá gốc
    'F': 12,   # Giá sale
    'G': 10,   # Tồn kho
    'H': 10,   # Scale Qty
    'I': 50,   # Link
}
for col, width in column_widths.items():
    ws.column_dimensions[col].width = width

# Freeze hàng đầu
ws.freeze_panes = "A2"

# 💾 Lưu file
output_file = "winmart_products.xlsx"
wb.save(output_file)
print(f"✅ Xong! Tổng {len(all_rows)} sản phẩm → {output_file}")

📦 Tổng sản phẩm đọc được: 6792
✅ Xong! Tổng 6792 sản phẩm → winmart_products.xlsx


In [1]:
import pandas as pd

df = pd.read_excel(r'D:\a\sp.xlsx')

print(f"📊 Tổng số dòng: {len(df)}")
print(f"📋 Các cột: {df.columns.tolist()}\n")
print("=" * 50)

for col in df.columns:
    print(f"\n🔹 Cột: [{col}]")
    print(f"   - Kiểu dữ liệu : {df[col].dtype}")
    print(f"   - Tổng giá trị : {len(df[col])}")
    print(f"   - Giá trị rỗng : {df[col].isna().sum()}")
    print(f"   - Giá trị duy nhất: {df[col].nunique()}")

    # Nếu là số thì in min/max/mean
    if df[col].dtype in ['int64', 'float64']:
        print(f"   - Min  : {df[col].min()}")
        print(f"   - Max  : {df[col].max()}")
        print(f"   - Trung bình: {df[col].mean():.2f}")
    else:
        # Nếu là text thì in 5 giá trị mẫu
        print(f"   - Mẫu  : {df[col].dropna().unique()[:5].tolist()}")

📊 Tổng số dòng: 3253
📋 Các cột: ['Danh mục (file)', 'Mã SP', 'Tên sản phẩm', 'Mô tả ngắn', 'Thương hiệu', 'Đơn vị', 'Barcode', 'Giá gốc', 'Giá sale', 'Tồn kho', 'Ngành hàng 1', 'Ngành hàng 2', 'Nhóm hàng 3', 'Nhóm hàng 4', 'Nhóm hàng 5', 'Category', 'Link']


🔹 Cột: [Danh mục (file)]
   - Kiểu dữ liệu : str
   - Tổng giá trị : 3253
   - Giá trị rỗng : 0
   - Giá trị duy nhất: 43
   - Mẫu  : ['banh-bao--c01156&storeCode=1535', 'banh-keo--c07&storeCode=1535', 'banh-mi--c0115&storeCode=1535', 'bia--c01135&storeCode=1535', 'binh-xit-con-trung--c01139&storeCode=1535']

🔹 Cột: [Mã SP]
   - Kiểu dữ liệu : int64
   - Tổng giá trị : 3253
   - Giá trị rỗng : 0
   - Giá trị duy nhất: 3253
   - Min  : 10005016
   - Max  : 10640565
   - Trung bình: 10185261.08

🔹 Cột: [Tên sản phẩm]
   - Kiểu dữ liệu : str
   - Tổng giá trị : 3253
   - Giá trị rỗng : 0
   - Giá trị duy nhất: 3248
   - Mẫu  : [' Bánh bao nhân thịt heo trứng cút Thọ Phát gói 250g ', ' Bánh bao Malai nhân xíu mại thịt heo 300g', ' Bán